In [1]:
import pandas as pd
import glob
import os
import json

# ==========================================
# STEP 1: CONFIGURATION & HELPERS
# ==========================================
RESULTS_DIR = "data/"
OUTPUT_DIR = "analysis_output/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def get_family(model_name):
    lower = model_name.lower()
    if "pythia" in lower: return "Pythia"
    if "qwen" in lower: return "Qwen 2.5"
    if "bloom" in lower: return "BLOOM"
    if "gemma" in lower: return "Gemma 3"
    if "llama" in lower: return "Llama 3.2"
    return "Other"

def load_clean_csvs(pattern):
    files = glob.glob(os.path.join(RESULTS_DIR, pattern))
    if not files: raise FileNotFoundError(f"No files found for {pattern}")
    
    dfs = []
    for f in files:
        df = pd.read_csv(f)
        # Fallback to extract model name from filename if missing (e.g., validation CSVs)
        if 'model' not in df.columns:
            base_name = os.path.basename(f)
            clean_name = base_name.replace("validation_", "").replace("erasure_", "").replace(".csv", "")
            clean_name = clean_name.split("_seed")[0]
            df['model'] = clean_name
        dfs.append(df)
        
    df = pd.concat(dfs, ignore_index=True)
    df['model'] = df['model'].str.replace("/", "_")
    df['family'] = df['model'].apply(get_family)
    return df

# ==========================================
# STEP 2: LOAD & FILTER (~3B CAPACITY ONLY)
# ==========================================
print("Loading CSVs...")
df_bpb_raw = load_clean_csvs("erasure_*.csv")
df_val_raw = load_clean_csvs("validation_*.csv")

# Strict 3B Class Filter to isolate Pre-Training Distribution
valid_3b_models = [
    "meta-llama_Llama-3.2-3B",
    "EleutherAI_pythia-2.8b",
    "Qwen_Qwen2.5-3B",
    "bigscience_bloom-3b",
    "google_gemma-3-4b-it"
]

df_bpb = df_bpb_raw[df_bpb_raw['model'].isin(valid_3b_models)].copy()
df_val = df_val_raw[df_val_raw['model'].isin(valid_3b_models)].copy()

print(f"Filtered BPB Rows (~3B only): {len(df_bpb)}")
print(f"Filtered Validation Rows (~3B only): {len(df_val)}")

# ==========================================
# STEP 3: COMPUTE BASE METRICS (TABLE I)
# ==========================================
df_base_bpb = df_bpb[df_bpb['erased_concept'] == 'baseline']
base_bpb_summary = df_base_bpb.groupby(['family', 'model'])['bpb'].mean().reset_index(name='Mean_Base_BPB')

df_base_probe = df_val[['family', 'model', 'layer', 'seed', 'probe_acc_before']].drop_duplicates()
mean_probe_per_seed = df_base_probe.groupby(['family', 'model', 'seed'])['probe_acc_before'].mean().reset_index()
base_probe_summary = mean_probe_per_seed.groupby(['family', 'model'])['probe_acc_before'].agg(
    Mean_Base_Probe_Acc='mean',
    Standard_Error='sem'
).reset_index()

# ==========================================
# STEP 4: COMPUTE DELTA METRICS (TABLES II-IV)
# ==========================================
df_bpb_targets = df_bpb[~df_bpb['erased_concept'].isin(['baseline', 'ind'])].copy()
df_val_targets = df_val[~df_val['erased_concept'].isin(['baseline', 'ind'])].copy()

bpb_delta_summary = df_bpb_targets.groupby(['family', 'erased_concept'])['bpb_delta'].agg(
    Mean_BPB_Delta='mean',
    Standard_Error='sem'
).reset_index()

# Structural Impact: Calculate actual delta (inverting the drop)
df_val_targets['probe_acc_delta'] = -df_val_targets['probe_acc_drop']
probe_delta_summary = df_val_targets.groupby(['family', 'erased_concept'])['probe_acc_delta'].agg(
    Mean_Probe_Acc_Delta='mean',
    Standard_Error='sem'
).reset_index()

# ==========================================
# STEP 5: EXPORT TO JSON
# ==========================================
base_out_path = os.path.join(OUTPUT_DIR, "table_1_base_metrics.json")
with open(base_out_path, "w") as f:
    json.dump({
        "base_bpb": base_bpb_summary.to_dict(orient="records"),
        "mean_base_linear_separability": base_probe_summary.to_dict(orient="records")
    }, f, indent=4)

delta_out_path = os.path.join(OUTPUT_DIR, "tables_2_3_delta_metrics.json")
with open(delta_out_path, "w") as f:
    json.dump({
        "bpb_generative_impact": bpb_delta_summary.to_dict(orient="records"),
        "layer_separability_impact": probe_delta_summary.to_dict(orient="records")
    }, f, indent=4)

print("\n[SUCCESS] Pipeline Complete. Clean JSON files generated in output directory:")
print(f"1. {base_out_path}")
print(f"2. {delta_out_path}")

Loading CSVs...
Filtered BPB Rows (~3B only): 2000
Filtered Validation Rows (~3B only): 1750

[SUCCESS] Pipeline Complete. Clean JSON files generated in output directory:
1. analysis_output/table_1_base_metrics.json
2. analysis_output/tables_2_3_delta_metrics.json
